# NHS England Maternal Care Pathways Master Pipeline
## Stage 2.2 - Load HES_APC data and map to MSDS data

### Compiled by Ethan Phillips (Oxford)

### Last updated 10.11.2025

In [0]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyspark.sql import functions as F
from pyspark.sql import SparkSession 
from pyspark.sql.functions import when, col, lit, count, max, last, first, min, avg, sum, collect_list, collect_set, countDistinct, to_date, datediff, array_contains, size, udf, format_number, regexp_replace, explode, array, array_union, desc_nulls_last
from pyspark.sql.types import IntegerType, StringType, NumericType, DateType, ArrayType
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.functions import vector_to_array
from pyspark.sql.window import Window
from pyspark.ml.regression import GeneralizedLinearRegression as GLR


In [0]:
spark = SparkSession.builder.getOrCreate()

In [0]:
read_table_name = "msds_demographics_agg_by_uniqpregid_stage"

df_demographics_agg = spark.read.table(f"dsa_712819_x8g2j.dsa_712819_x8g2j_collab.{read_table_name}")

In [0]:
df_hes_apc = spark.sql(
    "SELECT DISTINCT * FROM dsa_712819_x8g2j.dsa_712819_x8g2j_collab.hes_apc_all_years "
    "WHERE EPITYPE IN (2, 3, 5, 6)"
)
df_hes_apc_mat = spark.sql(
    "SELECT DISTINCT * FROM dsa_712819_x8g2j.dsa_712819_x8g2j_collab.hes_apc_mat_all_years"
).drop(col("FYEAR"))

# There is one-to-one correspondence between EPIKEYs in df_hes_apc and df_hes_apc_mat (no additional entries in either dataset)

# Merge columns from df_hes_apc_mat into df_hes_apc on EPIKEY and PERSON_ID_DEID
df_hes_apc = df_hes_apc.join(df_hes_apc_mat, on=["EPIKEY","PERSON_ID_DEID"], how="left")

In [0]:
df_hes_apc = df_hes_apc.withColumns({
    "ANASDATE": to_date("ANASDATE", "yyyy-MM-dd"),
    "ADMIDATE": to_date("ADMIDATE", "yyyy-MM-dd"),
    "DISDATE": to_date("DISDATE", "yyyy-MM-dd"),

    "ANAGEST": col("ANAGEST").try_cast('int'),
    "STARTAGE": col("STARTAGE").try_cast('int'),
    "BIRSTAT_1": col("BIRSTAT_1").try_cast('int'),
    "BIRWEIT_1": col("BIRWEIT_1").try_cast('int'),
    "DELMETH_D": col("DELMETH_D").try_cast('int'),
    "GESTAT_1": col("GESTAT_1").try_cast('int'),
    "MATAGE": col("MATAGE").try_cast('int'),
    "NUMBABY": col("NUMBABY").try_cast('int'),
    "NUMTAILB": col("NUMTAILB").try_cast('int'),
    "SEXBABY_1": col("SEXBABY_1").try_cast('int'),
}).groupBy("PERSON_ID_DEID", "ADMIDATE").agg(
    first("FYEAR", ignorenulls=True).alias("FYEAR"),
    first("EPIKEY", ignorenulls=True).alias("EPIKEY"),

    min("STARTAGE").alias("STARTAGE"),
    min("MATAGE").alias("MATAGE"),
    min("ANASDATE").alias("ANASDATE"),
    max("DISDATE").alias("DISDATE"),
    max("NUMBABY").alias("NUMBABY"),
    last("SITETRET", ignorenulls=True).alias("SITETRET"),

    first("BIRSTAT_1", ignorenulls=True).alias("BIRSTAT"),
    first("BIRWEIT_1", ignorenulls=True).alias("BIRWEIT"),
    first("DELMETH_D", ignorenulls=True).alias("DELMETH"),
    first("ANAGEST", ignorenulls=True).alias("ANAGEST"),
    first("GESTAT_1", ignorenulls=True).alias("GESTAT"),
    first("SEXBABY_1", ignorenulls=True).alias("SEXBABY")
).withColumn(
    "BIRYEAR", F.year(col("ADMIDATE"))
)


In [0]:
# Prep reduced datasets
hes_epikeys = (
    df_hes_apc
    .filter(~col("ANASDATE").isNull())
    .select([
        col("PERSON_ID_DEID").alias("hes_id"), 
        col("ANASDATE").alias("hes_ante_appt_date"), 
        col("EPIKEY"),
        col("ADMIDATE").alias("hes_ld_admit_date")
    ])
)
df_master_pregs = (
    df_demographics_agg
    .filter(~col("antenatal_appt_date").isNull())
    .select([
        "person_id_deid",
        "uniqpregid",
        "antenatal_appt_date",
        "ld_hosp_start_date"
    ])
)

# Join master pregnancy records to HES epikeys on exact person ID match
# and where the HES appointment date is within ±3 days of the antenatal appointment date
msds_hes_map = (
    df_master_pregs
    .join(
        hes_epikeys,
        (col("person_id_deid") == col("hes_id")) & \
        (
            (
                (~col("hes_ante_appt_date").isNull()) & \
                (F.abs(F.datediff(col("antenatal_appt_date"), col("hes_ante_appt_date"))) <= 3)
            ) | (
                (~col("ld_hosp_start_date").isNull()) & \
                (F.abs(F.datediff(col("ld_hosp_start_date"), col("hes_ld_admit_date"))) <= 3)
            )
        ),
        how="left"
    )
    .select([
        "person_id_deid",
        "uniqpregid",
        "epikey",
        "antenatal_appt_date",
        "hes_ld_admit_date"
    ])
)

# Count epikeys per pregnancy and show distribution
epikey_counts = (
    msds_hes_map
    .groupBy("uniqpregid")
    .agg(F.count(F.col("epikey")).alias("epikey_cnt"))
)

epikey_distribution = (
    epikey_counts
    .groupBy("epikey_cnt")
    .agg(F.count("*").alias("num_uniqpregid"))
    .orderBy("epikey_cnt")
)

display(epikey_distribution)

# Keep only the epikey with the latest hes_ld_admit_date for each uniqpregid
msds_hes_map = (
    msds_hes_map
    .withColumn(
        "rn",
        F.row_number().over(
            Window.partitionBy("uniqpregid")
                  .orderBy(col("hes_ld_admit_date").desc_nulls_last())
        )
    )
    .filter(col("rn") == 1)
    .drop("rn")
)

In [0]:
# Compute epikeys that map to exactly one uniqpregid and keep only those rows
epikey_unique = (
    msds_hes_map
    .groupBy("epikey")
    .agg(F.countDistinct("uniqpregid").alias("uniqpregid_cnt"))
    .filter(col("uniqpregid_cnt") == 1)
    .select("epikey")
)

# Retain rows whose epikey is unique across uniqpregids
# msds_hes_map = msds_hes_map.join(epikey_unique, on="epikey", how="inner")

In [0]:
df_msds_hes = msds_hes_map.select("uniqpregid", "epikey")

In [0]:
# MSDS (UniqPregID) and HES (epikey) mapping 
write_table_name_1 = "msds_hes_mapping_stage"

df_msds_hes.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"dsa_712819_x8g2j.dsa_712819_x8g2j_collab.{write_table_name_1}")

# Aggregated HES data
write_table_name_2 = "hes_aggregated_LD_spells_stage"

df_hes_apc.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"dsa_712819_x8g2j.dsa_712819_x8g2j_collab.{write_table_name_2}")